# raw_crop_wx pooled LightGBM：資本・生産要素グループ別 OOF SHAP

現在の最終候補である raw_crop_wx の単一・pooled二段階LightGBMについて、
個別36変数と、生産要素の大分類別にOOF SHAPを計算する。
変数処理と周辺変数は spatial_correration.ipynb のraw_crop_wx再推定と揃える。

## セル1：分析方針

- LightGBMは36個の個別変数をそのまま使う
- SHAPの解釈段階だけを大分類にまとめる
- Stage 1とStage 2を分けて表示する
- 5-fold空間OOFでテストデータのSHAPを計算する
- 個別セル別SHAPも保存する
- data_availabilityは生産要素ではないため補助表示にする

大分類は、労働・人的資本、気候・地理・自然資本、
市場・インフラ資本、技術・投入資本の4つである。

## セル2：データ読み込み

In [1]:
from __future__ import annotations

import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)
from sklearn.neighbors import BallTree
from lightgbm import LGBMClassifier, LGBMRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path(r"C:\masterresearch\Comparative_advantage")
GAEZ_DIR = ROOT / "GAEZ"
HYDE_DIR = ROOT / "HYDE3.4"
CROPLAND_DIR = HYDE_DIR / "cropland_npys"
DIST_DIR = ROOT / "distance_to_cities"
GLOFAS_DIR = ROOT / "GloFAS" / "processed_5min"
FEATURE_CACHE = GAEZ_DIR / "CroplandRegression" / "features_cache"
OUTPUT_DIR = GAEZ_DIR / "CroplandRegression" / "spatial_wx_comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
YEAR = 2024
PRESENCE_THRESHOLD = 0.01
N_POS_SAMPLE = 120_000
N_ZERO_SAMPLE = 120_000
N_SPLITS = 5

WX_RADII_KM = (50, 100)
WX_SOURCE_FEATURES = [
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_rainfed_value_top5",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_2020",
]
INCLUDE_SLOPE_WX = False
MIN_VALID_NEIGHBOR_FRACTION = 0.25
SAVE_WX_RASTERS = True

EARTH_RADIUS_KM = 6371.0088
MORAN_K = 8
MORAN_MAX_N = 50_000


def load_cache(*names):
    for name in names:
        path = FEATURE_CACHE / name
        if path.exists():
            print("load cache:", path.name)
            return np.load(path, mmap_mode="r")
    raise FileNotFoundError(
        "Required cache not found: " + ", ".join(names)
    )


def safe_log1p(values):
    values = np.asarray(values, dtype=np.float32)
    values = np.where(np.isfinite(values) & (values > 0), values, 0.0)
    return np.log1p(values).astype(np.float32)


lat = np.load(CROPLAND_DIR / "lat.npy")
lon = np.load(CROPLAND_DIR / "lon.npy")
SHAPE = (len(lat), len(lon))

cropland_raw = np.load(
    CROPLAND_DIR / "cropland_fraction_1950_2024.npy",
    mmap_mode="r",
)
years = np.load(CROPLAND_DIR / "years.npy")
year_index = int(np.where(years == YEAR)[0][0])
cropland_2024 = np.asarray(cropland_raw[year_index], dtype=np.float32).copy()
cropland_2024[~np.isfinite(cropland_2024)] = np.nan

population_density_2024 = load_cache("population_density_2024.npy")
elevation_m = load_cache("elevation_5min.npy")
slope = load_cache("slope_5min.npy")
exclusion = load_cache("exclusion_5min_mode.npy")
city_time_20k_min = np.load(DIST_DIR / "cities_10_1_12deg_min.npy", mmap_mode="r")
port_time_any_min = np.load(DIST_DIR / "ports_05_1_12deg_min.npy", mmap_mode="r")
glofas_p10 = np.load(GLOFAS_DIR / "p10_discharge_max_5min_2020.npy", mmap_mode="r")
distance_river_gt10 = np.load(
    GLOFAS_DIR / "distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy",
    mmap_mode="r",
)
rainfed_value_top5 = load_cache(
    "rainfed_value_top5_usd_per_ha_checked_36crops.npy",
    "rainfed_value_top5_checked_36crops.npy",
)
irrigated_value_top5 = load_cache(
    "irrigated_value_top5_usd_per_ha_checked_36crops.npy",
    "irrigated_value_top5_checked_36crops.npy",
)
rainfed_calorie_top5 = load_cache(
    "rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "rainfed_calorie_top5_checked_36crops.npy",
)
irrigated_calorie_top5 = load_cache(
    "irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy",
    "irrigated_calorie_top5_checked_36crops.npy",
)

land_mask = (
    np.isfinite(elevation_m)
    & np.isfinite(cropland_2024)
    & np.isfinite(population_density_2024)
    & (population_density_2024 >= 0)
)
presence = land_mask & (cropland_2024 > PRESENCE_THRESHOLD)

rng = np.random.default_rng(RANDOM_SEED)
pos_flat = np.flatnonzero(presence.ravel())
zero_flat = np.flatnonzero((land_mask & ~presence).ravel())
pos_sample = rng.choice(pos_flat, min(N_POS_SAMPLE, len(pos_flat)), replace=False)
zero_sample = rng.choice(zero_flat, min(N_ZERO_SAMPLE, len(zero_flat)), replace=False)
sample_flat = np.concatenate([pos_sample, zero_sample])
rng.shuffle(sample_flat)
rows, cols = np.unravel_index(sample_flat, SHAPE)


def take(array):
    return np.asarray(array[rows, cols])


sample = pd.DataFrame({
    "row": rows.astype(np.int32),
    "col": cols.astype(np.int32),
    "lat": lat[rows].astype(np.float32),
    "lon": lon[cols].astype(np.float32),
    "cropland_fraction": take(cropland_2024).astype(np.float32),
    "presence": (take(cropland_2024) > PRESENCE_THRESHOLD).astype(np.uint8),
    "elevation_m": take(elevation_m).astype(np.float32),
    "slope": take(slope).astype(np.float32),
    "exclusion_class": np.nan_to_num(take(exclusion), nan=-1).astype(np.int16),
    "log_pop_density_2024": safe_log1p(take(population_density_2024)),
    "log_city_time_20k_min": safe_log1p(take(city_time_20k_min)),
    "log_port_time_any_min": safe_log1p(take(port_time_any_min)),
    "log_glofas_p10_2020": safe_log1p(take(glofas_p10)),
    "log_distance_river_gt10_2020": safe_log1p(take(distance_river_gt10)),
    "log_rainfed_value_top5": safe_log1p(take(rainfed_value_top5)),
    "log_rainfed_calorie_top5": safe_log1p(take(rainfed_calorie_top5)),
    "log_irrigation_value_gain_top5": safe_log1p(
        np.maximum(take(irrigated_value_top5) - take(rainfed_value_top5), 0)
    ),
    "log_irrigation_calorie_gain_top5": safe_log1p(
        np.maximum(take(irrigated_calorie_top5) - take(rainfed_calorie_top5), 0)
    ),
}).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

sample["spatial_block"] = (
    np.floor((sample["lat"] + 90) / 10).astype(int) * 36
    + np.floor((sample["lon"] + 180) / 10).astype(int)
)

base_feature_cols = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_2020",
    "log_rainfed_value_top5",
    "log_rainfed_calorie_top5",
    "log_irrigation_value_gain_top5",
    "log_irrigation_calorie_gain_top5",
]

print("grid:", SHAPE)
print("sample rows:", len(sample))
print("sample presence share:", float(sample["presence"].mean()))

load cache: population_density_2024.npy
load cache: elevation_5min.npy
load cache: slope_5min.npy
load cache: exclusion_5min_mode.npy
load cache: rainfed_value_top5_usd_per_ha_checked_36crops.npy
load cache: irrigated_value_top5_usd_per_ha_checked_36crops.npy
load cache: rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy
load cache: irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy
grid: (2160, 4320)
sample rows: 240000
sample presence share: 0.5


## セル3：周辺変数WXの準備

In [2]:
def _rolling_sum_horizontal(values, half_width):
    values = np.asarray(values, dtype=np.float32)
    if half_width <= 0:
        return values.copy()
    extended = np.concatenate(
        [values[:, -half_width:], values, values[:, :half_width]],
        axis=1,
    )
    width = 2 * half_width + 1
    cumulative = np.concatenate(
        [
            np.zeros((values.shape[0], 1), dtype=np.float64),
            np.cumsum(extended, axis=1, dtype=np.float64),
        ],
        axis=1,
    )
    return (cumulative[:, width:] - cumulative[:, :-width]).astype(np.float32)


def _rolling_sum_vertical(values, half_width):
    values = np.asarray(values, dtype=np.float32)
    if half_width <= 0:
        return values.copy()
    padded = np.pad(values, ((half_width, half_width), (0, 0)), mode="edge")
    width = 2 * half_width + 1
    cumulative = np.concatenate(
        [
            np.zeros((1, values.shape[1]), dtype=np.float64),
            np.cumsum(padded, axis=0, dtype=np.float64),
        ],
        axis=0,
    )
    return (cumulative[width:, :] - cumulative[:-width, :]).astype(np.float32)


def latitude_aware_box_mean(values, lat_values, radius_km, min_valid_fraction=0.25):
    values = np.asarray(values, dtype=np.float32)
    if values.shape != SHAPE:
        raise ValueError(f"Unexpected raster shape: {values.shape}; expected {SHAPE}")

    lat_step_deg = float(np.median(np.abs(np.diff(lat_values))))
    lon_step_deg = float(np.median(np.abs(np.diff(lon))))
    km_per_degree = EARTH_RADIUS_KM * np.pi / 180.0
    half_lat = max(1, int(np.ceil(radius_km / (km_per_degree * lat_step_deg))))

    cos_lat = np.clip(np.cos(np.deg2rad(lat_values)), 1e-6, None)
    half_lon_by_row = np.ceil(
        radius_km / (km_per_degree * cos_lat * lon_step_deg)
    ).astype(np.int32)
    half_lon_by_row = np.minimum(half_lon_by_row, values.shape[1] // 2 - 1)

    finite = np.isfinite(values)
    filled = np.where(finite, values, 0).astype(np.float32)
    counts = finite.astype(np.float32)
    horizontal_sum = np.zeros_like(filled, dtype=np.float32)
    horizontal_count = np.zeros_like(counts, dtype=np.float32)

    for half_lon in np.unique(half_lon_by_row):
        row_indices = np.flatnonzero(half_lon_by_row == half_lon)
        horizontal_sum[row_indices] = _rolling_sum_horizontal(
            filled[row_indices], int(half_lon)
        )
        horizontal_count[row_indices] = _rolling_sum_horizontal(
            counts[row_indices], int(half_lon)
        )

    vertical_sum = _rolling_sum_vertical(horizontal_sum, half_lat)
    vertical_count = _rolling_sum_vertical(horizontal_count, half_lat)

    row_width = (2 * half_lon_by_row + 1).astype(np.float64)
    padded_width = np.pad(row_width, (half_lat, half_lat), mode="edge")
    cumulative_width = np.concatenate([[0.0], np.cumsum(padded_width)])
    window_width_by_row = (
        cumulative_width[2 * half_lat + 1:]
        - cumulative_width[:-(2 * half_lat + 1)]
    )
    minimum_count = min_valid_fraction * window_width_by_row

    result = np.full(values.shape, np.nan, dtype=np.float32)
    valid = vertical_count >= minimum_count[:, None]
    with np.errstate(divide="ignore", invalid="ignore"):
        result[valid] = (vertical_sum[valid] / vertical_count[valid]).astype(np.float32)
    return result


wx_source_rasters = {
    "log_pop_density_2024": population_density_2024,
    "log_city_time_20k_min": city_time_20k_min,
    "log_port_time_any_min": port_time_any_min,
    "log_rainfed_value_top5": rainfed_value_top5,
    "log_glofas_p10_2020": glofas_p10,
    "log_distance_river_gt10_2020": distance_river_gt10,
}
if INCLUDE_SLOPE_WX:
    wx_source_rasters["slope"] = slope
    WX_SOURCE_FEATURES = WX_SOURCE_FEATURES + ["slope"]

sample_rows = sample["row"].to_numpy(dtype=int)
sample_cols = sample["col"].to_numpy(dtype=int)
wx_feature_cols = []

for radius_km in WX_RADII_KM:
    for source_name in WX_SOURCE_FEATURES:
        wx_name = f"wx_{radius_km}km_{source_name}"
        cache_path = OUTPUT_DIR / f"{wx_name}.npy"

        if cache_path.exists():
            print("load WX cache:", cache_path.name)
            wx_raster = np.load(cache_path, mmap_mode="r")
        else:
            print("build WX:", wx_name)
            if source_name == "slope":
                source_values = np.asarray(wx_source_rasters[source_name], dtype=np.float32)
            else:
                source_values = safe_log1p(wx_source_rasters[source_name])
            wx_raster = latitude_aware_box_mean(
                source_values,
                lat,
                radius_km,
                min_valid_fraction=MIN_VALID_NEIGHBOR_FRACTION,
            )
            if SAVE_WX_RASTERS:
                np.save(cache_path, wx_raster.astype(np.float32, copy=False))

        sample[wx_name] = np.asarray(
            wx_raster[sample_rows, sample_cols],
            dtype=np.float32,
        )
        wx_feature_cols.append(wx_name)

        if "source_values" in locals():
            del source_values
        if not SAVE_WX_RASTERS and not isinstance(wx_raster, np.memmap):
            del wx_raster
        gc.collect()

print("WX feature count:", len(wx_feature_cols))

load WX cache: wx_50km_log_pop_density_2024.npy
load WX cache: wx_50km_log_city_time_20k_min.npy
load WX cache: wx_50km_log_port_time_any_min.npy
load WX cache: wx_50km_log_rainfed_value_top5.npy
load WX cache: wx_50km_log_glofas_p10_2020.npy
load WX cache: wx_50km_log_distance_river_gt10_2020.npy
load WX cache: wx_100km_log_pop_density_2024.npy
load WX cache: wx_100km_log_city_time_20k_min.npy
load WX cache: wx_100km_log_port_time_any_min.npy
load WX cache: wx_100km_log_rainfed_value_top5.npy
load WX cache: wx_100km_log_glofas_p10_2020.npy
load WX cache: wx_100km_log_distance_river_gt10_2020.npy
WX feature count: 12


## セル4：Spatial OOFの準備

In [3]:
sample = (
    sample.replace([np.inf, -np.inf], np.nan)
    .dropna(subset=base_feature_cols + wx_feature_cols)
    .reset_index(drop=True)
)
analysis_sample = sample.copy()
y_all = analysis_sample["presence"].to_numpy(dtype=np.uint8)
groups = analysis_sample["spatial_block"].to_numpy()

if analysis_sample["presence"].nunique() < 2:
    raise ValueError("Both presence classes are required.")
target_prior = float(presence.sum() / land_mask.sum())

feature_cols_by_model = {
    "baseline": base_feature_cols,
    "wx": base_feature_cols + wx_feature_cols,
}

try:
    cell_area_km2 = np.load(CROPLAND_DIR / "grid_area_km2.npy", mmap_mode="r")
    sample_area = np.asarray(
        cell_area_km2[
            analysis_sample["row"].to_numpy(int),
            analysis_sample["col"].to_numpy(int),
        ],
        dtype=float,
    )
    n_pos_pop = int(presence.sum())
    n_zero_pop = int(land_mask.sum() - presence.sum())
    n_pos_samp = int((y_all == 1).sum())
    n_zero_samp = int((y_all == 0).sum())
    area_weight = np.where(
        y_all == 1,
        n_pos_pop / max(n_pos_samp, 1),
        n_zero_pop / max(n_zero_samp, 1),
    ) * sample_area
except FileNotFoundError:
    print("grid_area_km2.npy not found; area weighting skipped.")
    area_weight = None

splits = list(
    GroupKFold(n_splits=N_SPLITS).split(
        analysis_sample,
        y_all,
        groups=groups,
    )
)
print("analysis rows:", len(analysis_sample))
print("spatial blocks:", analysis_sample["spatial_block"].nunique())
print("target prior:", target_prior)

analysis rows: 240000
spatial blocks: 291
target prior: 0.3658932992121897


## セル5：raw crop potential変数の作成

crop potentialはraw値を使い、存在・欠測フラグと灌漑の符号付きdeltaを追加する。

In [4]:
raw_analysis_sample = analysis_sample.copy()

raw_rows = raw_analysis_sample["row"].to_numpy(dtype=int)
raw_cols = raw_analysis_sample["col"].to_numpy(dtype=int)


def sample_raster(array):
    return np.asarray(
        array[raw_rows, raw_cols],
        dtype=np.float32,
    )


def clean_positive_feature(values):
    """
    正の値だけをraw値として残す。
    非正値は0にする。
    同時に、存在フラグと欠測フラグも返す。
    """
    values = np.asarray(values, dtype=np.float32)

    finite = np.isfinite(values)
    exists = finite & (values > 0)
    missing = ~finite

    clean = np.where(
        exists,
        values,
        0.0,
    ).astype(np.float32)

    return (
        clean,
        exists.astype(np.uint8),
        missing.astype(np.uint8),
    )


# ------------------------------------------------------------
# 2. crop potentialのraw値を作成
# ------------------------------------------------------------

rainfed_value_sample = sample_raster(rainfed_value_top5)
irrigated_value_sample = sample_raster(irrigated_value_top5)

rainfed_calorie_sample = sample_raster(rainfed_calorie_top5)
irrigated_calorie_sample = sample_raster(irrigated_calorie_top5)


(
    rainfed_value_raw,
    rainfed_value_exists,
    rainfed_value_missing,
) = clean_positive_feature(rainfed_value_sample)

(
    rainfed_calorie_raw,
    rainfed_calorie_exists,
    rainfed_calorie_missing,
) = clean_positive_feature(rainfed_calorie_sample)


# ------------------------------------------------------------
# 3. 灌漑gainと符号付き差分を作成
# ------------------------------------------------------------

value_pair_valid = (
    np.isfinite(rainfed_value_sample)
    & np.isfinite(irrigated_value_sample)
)

calorie_pair_valid = (
    np.isfinite(rainfed_calorie_sample)
    & np.isfinite(irrigated_calorie_sample)
)


# 符号付き差分
irrigation_value_delta_signed = np.where(
    value_pair_valid,
    irrigated_value_sample - rainfed_value_sample,
    0.0,
).astype(np.float32)

irrigation_calorie_delta_signed = np.where(
    calorie_pair_valid,
    irrigated_calorie_sample - rainfed_calorie_sample,
    0.0,
).astype(np.float32)


# 非負gain
irrigation_value_gain_raw = np.maximum(
    irrigation_value_delta_signed,
    0.0,
).astype(np.float32)

irrigation_calorie_gain_raw = np.maximum(
    irrigation_calorie_delta_signed,
    0.0,
).astype(np.float32)


# gainの存在フラグ
irrigation_value_gain_exists = (
    value_pair_valid
    & (irrigation_value_delta_signed > 0)
).astype(np.uint8)

irrigation_calorie_gain_exists = (
    calorie_pair_valid
    & (irrigation_calorie_delta_signed > 0)
).astype(np.uint8)


# 欠測フラグ
irrigation_value_gain_missing = (
    ~value_pair_valid
).astype(np.uint8)

irrigation_calorie_gain_missing = (
    ~calorie_pair_valid
).astype(np.uint8)


# ------------------------------------------------------------
# 4. DataFrameに追加
# ------------------------------------------------------------

raw_analysis_sample[
    "rainfed_value_top5_raw"
] = rainfed_value_raw

raw_analysis_sample[
    "rainfed_calorie_top5_raw"
] = rainfed_calorie_raw

raw_analysis_sample[
    "irrigation_value_gain_top5_raw"
] = irrigation_value_gain_raw

raw_analysis_sample[
    "irrigation_calorie_gain_top5_raw"
] = irrigation_calorie_gain_raw


raw_analysis_sample[
    "rainfed_value_top5_exists"
] = rainfed_value_exists

raw_analysis_sample[
    "rainfed_calorie_top5_exists"
] = rainfed_calorie_exists

raw_analysis_sample[
    "irrigation_value_gain_top5_exists"
] = irrigation_value_gain_exists

raw_analysis_sample[
    "irrigation_calorie_gain_top5_exists"
] = irrigation_calorie_gain_exists


raw_analysis_sample[
    "irrigation_value_delta_top5_signed"
] = irrigation_value_delta_signed

raw_analysis_sample[
    "irrigation_calorie_delta_top5_signed"
] = irrigation_calorie_delta_signed


raw_analysis_sample[
    "rainfed_value_top5_missing"
] = rainfed_value_missing

raw_analysis_sample[
    "rainfed_calorie_top5_missing"
] = rainfed_calorie_missing

raw_analysis_sample[
    "irrigation_value_gain_top5_missing"
] = irrigation_value_gain_missing

raw_analysis_sample[
    "irrigation_calorie_gain_top5_missing"
] = irrigation_calorie_gain_missing


# ------------------------------------------------------------
# 5. 新しいcrop potential feature list
# ------------------------------------------------------------

non_crop_base_cols = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_pop_density_2024",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_glofas_p10_2020",
    "log_distance_river_gt10_min",
]

# 実際の列名を確認
if "log_distance_river_gt10_min" not in raw_analysis_sample.columns:
    non_crop_base_cols[-1] = "log_distance_river_gt10_2020"


raw_crop_cols = [
    # rawの大きさ
    "rainfed_value_top5_raw",
    "rainfed_calorie_top5_raw",
    "irrigation_value_gain_top5_raw",
    "irrigation_calorie_gain_top5_raw",

    # 0か正値か
    "rainfed_value_top5_exists",
    "rainfed_calorie_top5_exists",
    "irrigation_value_gain_top5_exists",
    "irrigation_calorie_gain_top5_exists",

    # 灌漑による符号付き差分
    "irrigation_value_delta_top5_signed",
    "irrigation_calorie_delta_top5_signed",

    # 欠測フラグ
    "rainfed_value_top5_missing",
    "rainfed_calorie_top5_missing",
    "irrigation_value_gain_top5_missing",
    "irrigation_calorie_gain_top5_missing",
]


raw_crop_feature_cols = (
    non_crop_base_cols
    + raw_crop_cols
)


# ------------------------------------------------------------
# 6. raw版のcrop potential WXを作る
# ------------------------------------------------------------

raw_rainfed_value_raster = np.asarray(
    rainfed_value_top5,
    dtype=np.float32,
)

raw_rainfed_value_valid = np.isfinite(
    raw_rainfed_value_raster
)

raw_rainfed_value_clean_raster = np.where(
    raw_rainfed_value_valid
    & (raw_rainfed_value_raster > 0),
    raw_rainfed_value_raster,
    0.0,
).astype(np.float32)

raw_rainfed_value_exists_raster = (
    raw_rainfed_value_valid
    & (raw_rainfed_value_raster > 0)
).astype(np.float32)


raw_crop_wx_feature_cols = []

for radius_km in WX_RADII_KM:

    raw_wx_specs = [
        (
            f"wx_{radius_km}km_rainfed_value_top5_raw",
            raw_rainfed_value_clean_raster,
        ),
        (
            f"wx_{radius_km}km_rainfed_value_exists_share",
            raw_rainfed_value_exists_raster,
        ),
    ]

    for wx_name, source_values in raw_wx_specs:

        cache_path = (
            OUTPUT_DIR
            / f"{wx_name}.npy"
        )

        if cache_path.exists():
            print("load raw WX cache:", cache_path.name)
            wx_raster = np.load(
                cache_path,
                mmap_mode="r",
            )
        else:
            print("build raw WX:", wx_name)

            wx_raster = latitude_aware_box_mean(
                source_values,
                lat,
                radius_km,
                min_valid_fraction=MIN_VALID_NEIGHBOR_FRACTION,
            )

            if SAVE_WX_RASTERS:
                np.save(
                    cache_path,
                    wx_raster.astype(
                        np.float32,
                        copy=False,
                    ),
                )

        raw_analysis_sample[wx_name] = np.asarray(
            wx_raster[raw_rows, raw_cols],
            dtype=np.float32,
        )

        raw_crop_wx_feature_cols.append(wx_name)


# 既存WXから、log版rainfed value WXだけ除外
non_crop_wx_feature_cols = [
    col
    for col in wx_feature_cols
    if "log_rainfed_value_top5" not in col
]


raw_crop_wx_feature_cols = (
    raw_crop_feature_cols
    + non_crop_wx_feature_cols
    + raw_crop_wx_feature_cols
)


feature_cols_raw_by_model = {
    "raw_crop": raw_crop_feature_cols,
    "raw_crop_wx": raw_crop_wx_feature_cols,
}


print()
print("raw_crop feature count:", len(raw_crop_feature_cols))
print("raw_crop_wx feature count:", len(raw_crop_wx_feature_cols))
print()
print("raw_crop features:")
print(raw_crop_feature_cols)
print()
print("raw_crop_wx features:")
print(raw_crop_wx_feature_cols)


# ------------------------------------------------------------

load raw WX cache: wx_50km_rainfed_value_top5_raw.npy
load raw WX cache: wx_50km_rainfed_value_exists_share.npy
load raw WX cache: wx_100km_rainfed_value_top5_raw.npy
load raw WX cache: wx_100km_rainfed_value_exists_share.npy

raw_crop feature count: 22
raw_crop_wx feature count: 36

raw_crop features:
['elevation_m', 'slope', 'exclusion_class', 'log_pop_density_2024', 'log_city_time_20k_min', 'log_port_time_any_min', 'log_glofas_p10_2020', 'log_distance_river_gt10_2020', 'rainfed_value_top5_raw', 'rainfed_calorie_top5_raw', 'irrigation_value_gain_top5_raw', 'irrigation_calorie_gain_top5_raw', 'rainfed_value_top5_exists', 'rainfed_calorie_top5_exists', 'irrigation_value_gain_top5_exists', 'irrigation_calorie_gain_top5_exists', 'irrigation_value_delta_top5_signed', 'irrigation_calorie_delta_top5_signed', 'rainfed_value_top5_missing', 'rainfed_calorie_top5_missing', 'irrigation_value_gain_top5_missing', 'irrigation_calorie_gain_top5_missing']

raw_crop_wx features:
['elevation_m', 'slope

## セル6：raw_crop_wxの36変数を確認

In [5]:
RAW_WX_SHAP_MODEL = "raw_crop_wx"
RAW_WX_SHAP_FEATURES = list(feature_cols_raw_by_model[RAW_WX_SHAP_MODEL])
print("model:", RAW_WX_SHAP_MODEL)
print("feature count:", len(RAW_WX_SHAP_FEATURES))
print(RAW_WX_SHAP_FEATURES)

model: raw_crop_wx
feature count: 36
['elevation_m', 'slope', 'exclusion_class', 'log_pop_density_2024', 'log_city_time_20k_min', 'log_port_time_any_min', 'log_glofas_p10_2020', 'log_distance_river_gt10_2020', 'rainfed_value_top5_raw', 'rainfed_calorie_top5_raw', 'irrigation_value_gain_top5_raw', 'irrigation_calorie_gain_top5_raw', 'rainfed_value_top5_exists', 'rainfed_calorie_top5_exists', 'irrigation_value_gain_top5_exists', 'irrigation_calorie_gain_top5_exists', 'irrigation_value_delta_top5_signed', 'irrigation_calorie_delta_top5_signed', 'rainfed_value_top5_missing', 'rainfed_calorie_top5_missing', 'irrigation_value_gain_top5_missing', 'irrigation_calorie_gain_top5_missing', 'wx_50km_log_pop_density_2024', 'wx_50km_log_city_time_20k_min', 'wx_50km_log_port_time_any_min', 'wx_50km_log_glofas_p10_2020', 'wx_50km_log_distance_river_gt10_2020', 'wx_100km_log_pop_density_2024', 'wx_100km_log_city_time_20k_min', 'wx_100km_log_port_time_any_min', 'wx_100km_log_glofas_p10_2020', 'wx_100km_

## セル7：OOFモデルとTreeSHAP

In [6]:
import shap

SHAP_MAX_ROWS_PER_FOLD = 10_000
SHAP_OUTPUT_DIR = OUTPUT_DIR / "raw_crop_wx_capital_group_shap"
SHAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def fit_models(train_df, features, fold_number):
    classifier = LGBMClassifier(
        objective="binary", n_estimators=450, learning_rate=0.035,
        num_leaves=31, min_child_samples=80, subsample=0.85,
        colsample_bytree=0.85, random_state=RANDOM_SEED + fold_number,
        n_jobs=4, verbose=-1,
    )
    classifier.fit(
        train_df[features], train_df["presence"],
        categorical_feature=["exclusion_class"],
    )
    positive_train = train_df[train_df["presence"].eq(1)]
    regressor = LGBMRegressor(
        objective="regression", n_estimators=550, learning_rate=0.03,
        num_leaves=31, min_child_samples=60, subsample=0.85,
        colsample_bytree=0.85, random_state=RANDOM_SEED + fold_number,
        n_jobs=4, verbose=-1,
    )
    regressor.fit(
        positive_train[features],
        positive_train["cropland_fraction"],
        categorical_feature=["exclusion_class"],
    )
    return classifier, regressor


def choose_indices(indices, max_rows, seed):
    indices = np.asarray(indices, dtype=int)
    if max_rows is None or len(indices) <= max_rows:
        return np.sort(indices)
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(indices, size=max_rows, replace=False))


def tree_shap_raw(model, frame, features):
    values = shap.TreeExplainer(
        model,
        feature_perturbation="tree_path_dependent",
        model_output="raw",
    ).shap_values(frame[features], check_additivity=False)
    if isinstance(values, list):
        values = values[1] if len(values) == 2 else values[0]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, -1]
    if values.ndim != 2 or values.shape[1] != len(features):
        raise ValueError(f"Unexpected SHAP shape: {values.shape}")
    return values.astype(np.float32, copy=False)


def stats(values, indices):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(area_weight, dtype=float)[indices]
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[valid], weights[valid]
    if len(values) == 0:
        return {"n": 0, "weight_sum": 0.0, "abs_num": 0.0, "signed_num": 0.0}
    return {
        "n": int(len(values)),
        "weight_sum": float(weights.sum()),
        "abs_num": float(np.sum(weights * np.abs(values))),
        "signed_num": float(np.sum(weights * values)),
    }

## セル8：資本・生産要素の大分類

GloFAS、河川距離、天水ポテンシャルは大分類では気候・地理・自然資本に入れる。
灌漑ポテンシャルは技術・投入資本に入れる。

In [7]:
GROUP_ORDER = [
    "labor_human_capital",
    "climate_geographic_natural_capital",
    "market_infrastructure_capital",
    "technology_input_capital",
    "data_availability",
]
MAIN_GROUPS = GROUP_ORDER[:-1]

GROUP_LABELS_JA = {
    "labor_human_capital": "労働・人的資本",
    "climate_geographic_natural_capital": "気候・地理・自然資本",
    "market_infrastructure_capital": "市場・インフラ資本",
    "technology_input_capital": "技術・投入資本",
    "data_availability": "データ利用可能性",
}


def capital_group(feature):
    feature = str(feature)
    if "exists" in feature or "missing" in feature or "exists_share" in feature:
        return "data_availability"
    base = feature.split("km_", 1)[1] if feature.startswith("wx_") else feature
    if "pop_density" in base:
        return "labor_human_capital"
    if "city_time" in base or "port_time" in base:
        return "market_infrastructure_capital"
    if "irrigation" in base:
        return "technology_input_capital"
    if (
        base in {"elevation_m", "slope", "exclusion_class"}
        or "glofas" in base
        or "distance_river" in base
        or "rainfed" in base
    ):
        return "climate_geographic_natural_capital"
    raise ValueError(f"Unclassified feature: {feature}")


GROUP_MAP = {feature: capital_group(feature) for feature in RAW_WX_SHAP_FEATURES}
GROUP_FEATURES = {
    group: [feature for feature in RAW_WX_SHAP_FEATURES if GROUP_MAP[feature] == group]
    for group in GROUP_ORDER
}
group_definition = pd.DataFrame([
    {
        "group": group,
        "label_ja": GROUP_LABELS_JA[group],
        "n_features": len(GROUP_FEATURES[group]),
        "features": "|".join(GROUP_FEATURES[group]),
    }
    for group in GROUP_ORDER
])
display(group_definition)

                             group   label_ja  n_features                                                                                                                                                                                                                                                                                                                              features
               labor_human_capital    労働・人的資本           3                                                                                                                                                                                                                                                       log_pop_density_2024|wx_50km_log_pop_density_2024|wx_100km_log_pop_density_2024
climate_geographic_natural_capital 気候・地理・自然資本          13 elevation_m|slope|exclusion_class|log_glofas_p10_2020|log_distance_river_gt10_2020|rainfed_value_top5_raw|rainfed_calorie_top5_raw|wx_50km_log_glofas_p10_2020|wx_50km_log_dis

## セル9：5-fold OOF SHAPを計算

In [8]:
individual_rows = []
group_rows = []
local_presence_frames = []
local_fraction_frames = []


def add_row(rows, stage, fold, key, n_features, features, values, indices):
    s = stats(values, indices)
    rows.append({
        "model": RAW_WX_SHAP_MODEL,
        "stage": stage,
        "fold": fold,
        "key": key,
        "n_features": n_features,
        "features": "|".join(features),
        **s,
    })


for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
    print("=" * 70)
    print(f"RAW_CROP_WX SHAP FOLD {fold_number}/{N_SPLITS}")
    train_df = raw_analysis_sample.iloc[train_idx].copy()
    classifier, regressor = fit_models(
        train_df, RAW_WX_SHAP_FEATURES, fold_number
    )

    cls_idx = choose_indices(
        test_idx, SHAP_MAX_ROWS_PER_FOLD,
        RANDOM_SEED + 3000 + fold_number,
    )
    cls_frame = raw_analysis_sample.iloc[cls_idx].copy()
    cls_values = tree_shap_raw(
        classifier, cls_frame, RAW_WX_SHAP_FEATURES
    )

    for j, feature in enumerate(RAW_WX_SHAP_FEATURES):
        add_row(
            individual_rows, "presence_classifier", fold_number,
            feature, 1, [feature], cls_values[:, j], cls_idx,
        )

    for group in GROUP_ORDER:
        group_features = GROUP_FEATURES[group]
        idx = [RAW_WX_SHAP_FEATURES.index(f) for f in group_features]
        signed = cls_values[:, idx].sum(axis=1)
        abs_total = np.abs(cls_values[:, idx]).sum(axis=1)
        add_row(
            group_rows, "presence_classifier", fold_number,
            group, len(group_features), group_features,
            abs_total, cls_idx,
        )
        group_rows[-1]["signed_num"] = stats(signed, cls_idx)["signed_num"]

    local_cls = cls_frame[
        ["row", "col", "lat", "lon", "spatial_block"]
    ].copy()
    local_cls["cv_fold"] = fold_number
    for j, feature in enumerate(RAW_WX_SHAP_FEATURES):
        local_cls[f"shap__{feature}"] = cls_values[:, j]
    local_presence_frames.append(local_cls)

    pos_idx = test_idx[y_all[test_idx] == 1]
    pos_idx = choose_indices(
        pos_idx, SHAP_MAX_ROWS_PER_FOLD,
        RANDOM_SEED + 4000 + fold_number,
    )

    if len(pos_idx) > 0:
        pos_frame = raw_analysis_sample.iloc[pos_idx].copy()
        reg_values = tree_shap_raw(
            regressor, pos_frame, RAW_WX_SHAP_FEATURES
        )

        for j, feature in enumerate(RAW_WX_SHAP_FEATURES):
            add_row(
                individual_rows, "conditional_fraction_regressor", fold_number,
                feature, 1, [feature], reg_values[:, j], pos_idx,
            )

        for group in GROUP_ORDER:
            group_features = GROUP_FEATURES[group]
            idx = [RAW_WX_SHAP_FEATURES.index(f) for f in group_features]
            signed = reg_values[:, idx].sum(axis=1)
            abs_total = np.abs(reg_values[:, idx]).sum(axis=1)
            add_row(
                group_rows, "conditional_fraction_regressor", fold_number,
                group, len(group_features), group_features,
                abs_total, pos_idx,
            )
            group_rows[-1]["signed_num"] = stats(signed, pos_idx)["signed_num"]

        local_reg = pos_frame[
            ["row", "col", "lat", "lon", "spatial_block"]
        ].copy()
        local_reg["cv_fold"] = fold_number
        for j, feature in enumerate(RAW_WX_SHAP_FEATURES):
            local_reg[f"shap__{feature}"] = reg_values[:, j]
        local_fraction_frames.append(local_reg)

    del classifier, regressor, cls_values
    if "reg_values" in locals():
        del reg_values
    gc.collect()

print("OOF SHAP calculation finished.")

RAW_CROP_WX SHAP FOLD 1/5
C:\Users\tsuda\AppData\Local\Programs\Python\Python313\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
RAW_CROP_WX SHAP FOLD 2/5
C:\Users\tsuda\AppData\Local\Programs\Python\Python313\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
RAW_CROP_WX SHAP FOLD 3/5
C:\Users\tsuda\AppData\Local\Programs\Python\Python313\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
RAW_CROP_WX SHAP FOLD 4/5
C:\Users\tsuda\AppData\Local\Programs\Python\Python313\Lib\site-packages\shap\explainers\_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list 

## セル10：個別・大分類SHAPの集計と保存

In [9]:
individual_fold = pd.DataFrame(individual_rows)
group_fold = pd.DataFrame(group_rows)


def finalize(frame, key_name):
    result = (
        frame.groupby(
            ["model", "stage", "key", "n_features", "features"],
            as_index=False,
        )[["n", "weight_sum", "abs_num", "signed_num"]]
        .sum()
    )
    result["mean_abs_shap"] = result["abs_num"] / result["weight_sum"].replace(0, np.nan)
    result["mean_signed_shap"] = result["signed_num"] / result["weight_sum"].replace(0, np.nan)
    result["mean_abs_shap_per_feature"] = result["mean_abs_shap"] / result["n_features"].replace(0, np.nan)
    result["relative_importance_share"] = (
        result.groupby(["model", "stage"])["mean_abs_shap"]
        .transform(lambda values: values / values.sum())
    )
    return result.rename(columns={"key": key_name})


individual_summary = finalize(individual_fold, "feature")
group_summary = finalize(group_fold, "group")

group_definition.to_csv(
    SHAP_OUTPUT_DIR / "capital_group_definition.csv",
    index=False, encoding="utf-8-sig"
)
individual_fold.to_csv(
    SHAP_OUTPUT_DIR / "raw_crop_wx_individual_shap_fold_statistics.csv",
    index=False, encoding="utf-8-sig"
)
group_fold.to_csv(
    SHAP_OUTPUT_DIR / "raw_crop_wx_capital_group_shap_fold_statistics.csv",
    index=False, encoding="utf-8-sig"
)
individual_summary.to_csv(
    SHAP_OUTPUT_DIR / "raw_crop_wx_individual_shap_summary.csv",
    index=False, encoding="utf-8-sig"
)
group_summary.to_csv(
    SHAP_OUTPUT_DIR / "raw_crop_wx_capital_group_shap_summary.csv",
    index=False, encoding="utf-8-sig"
)

local_presence = pd.concat(local_presence_frames, ignore_index=True)
local_fraction = pd.concat(local_fraction_frames, ignore_index=True)
local_presence.to_csv(
    SHAP_OUTPUT_DIR / "raw_crop_wx_presence_local_shap.csv.gz",
    index=False, compression="gzip"
)
local_fraction.to_csv(
    SHAP_OUTPUT_DIR / "raw_crop_wx_conditional_fraction_local_shap.csv.gz",
    index=False, compression="gzip"
)
print("saved:", SHAP_OUTPUT_DIR)

saved: C:\masterresearch\Comparative_advantage\GAEZ\CroplandRegression\spatial_wx_comparison\raw_crop_wx_capital_group_shap


## セル11：個別変数・大分類をplot.showで表示

In [10]:
def plot_summary(frame, key, title, filename, value_column, selected=None):
    plot_frame = frame.copy()
    if selected is not None:
        plot_frame = plot_frame[plot_frame[key].isin(selected)].copy()

    stages = ["presence_classifier", "conditional_fraction_regressor"]
    fig, axes = plt.subplots(
        len(stages), 1, figsize=(15, 7 * len(stages)),
        squeeze=False, constrained_layout=True
    )

    for row_index, stage in enumerate(stages):
        stage_frame = plot_frame[
            plot_frame["stage"] == stage
        ].sort_values(value_column, ascending=True)
        ax = axes[row_index, 0]
        ax.barh(
            stage_frame[key].astype(str),
            stage_frame[value_column],
            color="#2563eb",
        )
        ax.set_title(f"{title}\n{stage}")
        ax.set_xlabel(value_column)
        ax.grid(axis="x", alpha=0.25)

    path = SHAP_OUTPUT_DIR / filename
    fig.savefig(path, dpi=220, bbox_inches="tight")
    plt.show()
    return path


individual_figure = plot_summary(
    individual_summary, "feature",
    "raw_crop_wx individual feature SHAP",
    "raw_crop_wx_individual_shap.png",
    "mean_abs_shap",
)
group_total_figure = plot_summary(
    group_summary, "group",
    "raw_crop_wx capital-group SHAP: total",
    "raw_crop_wx_capital_group_shap_total.png",
    "mean_abs_shap", MAIN_GROUPS,
)
group_per_feature_figure = plot_summary(
    group_summary, "group",
    "raw_crop_wx capital-group SHAP: per-feature normalized",
    "raw_crop_wx_capital_group_shap_per_feature.png",
    "mean_abs_shap_per_feature", MAIN_GROUPS,
)
data_figure = plot_summary(
    group_summary, "group",
    "raw_crop_wx data availability SHAP",
    "raw_crop_wx_data_availability_shap.png",
    "mean_abs_shap", ["data_availability"],
)

<string>:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
<string>:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
<string>:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
<string>:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [11]:
print("=" * 80)
print("INDIVIDUAL FEATURE SHAP")
print("=" * 80)
display(
    individual_summary[
        ["stage", "feature", "mean_abs_shap",
         "mean_signed_shap", "relative_importance_share"]
    ].round(6)
)

print("=" * 80)
print("CAPITAL-GROUP SHAP")
print("=" * 80)
display(
    group_summary[
        ["stage", "group", "n_features",
         "mean_abs_shap", "mean_abs_shap_per_feature",
         "mean_signed_shap", "relative_importance_share"]
    ].round(6)
)

print("保存先:", SHAP_OUTPUT_DIR)
print("個別変数図:", individual_figure)
print("大分類・合計図:", group_total_figure)
print("大分類・変数数補正図:", group_per_feature_figure)
print("data availability図:", data_figure)

INDIVIDUAL FEATURE SHAP
                         stage                               feature  mean_abs_shap  mean_signed_shap  relative_importance_share
conditional_fraction_regressor                           elevation_m       0.022228         -0.003259                   0.064222
conditional_fraction_regressor                       exclusion_class       0.029786          0.001035                   0.086057
conditional_fraction_regressor  irrigation_calorie_delta_top5_signed       0.005630         -0.000317                   0.016267
conditional_fraction_regressor   irrigation_calorie_gain_top5_exists       0.000131          0.000036                   0.000380
conditional_fraction_regressor  irrigation_calorie_gain_top5_missing       0.000000          0.000000                   0.000000
conditional_fraction_regressor      irrigation_calorie_gain_top5_raw       0.004806          0.000022                   0.013887
conditional_fraction_regressor    irrigation_value_delta_top5_signed     

## セル12：読み方

mean_abs_shapはグループ全体へのモデル依存度で、変数数が多いグループほど大きくなりやすい。
そのため、mean_abs_shap_per_featureも併記する。

mean_signed_shapは、グループが予測を押し上げる方向か押し下げる方向かを見る指標である。
data_availabilityは生産要素ではないので、農業レジームの解釈には直接使わない。

このノートブックのLightGBMは1本のpooledモデルであり、レジーム別モデルではない。
保存したセル別OOF SHAPは、次の段階で生産要素への反応プロファイルを分析するために使える。